In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score ,recall_score, f1_score,  classification_report
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../AI_PBL_data/web_server_logs_2.csv')
df

,ip,timestamp,method,status_code,size
0,192.168.1.138,2024-11-30 9:26,OPTIONS,301,541
1,192.168.1.130,2024-11-30 12:00,OPTIONS,200,5239
2,192.168.1.75,2024-12-01 0:22,POST,200,5778
3,192.168.1.176,2024-11-30 19:33,DELETE,200,3911
4,10.0.0.113,2024-11-30 5:37,GET,200,7765
...,...,...,...,...,...
1495,192.168.1.65,2024-11-30 5:06,GET,503,962
1496,192.168.1.51,2024-11-30 17:12,OPTIONS,200,6500
1497,10.0.0.116,2024-11-30 21:50,OPTIONS,200,3854
1498,192.168.1.2,2024-11-30 8:23,GET,404,4338


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   ip           1500 non-null   str  
 1   timestamp    1500 non-null   str  
 2   method       1500 non-null   str  
 3   status_code  1500 non-null   int64
 4   size         1500 non-null   int64
dtypes: int64(2), str(3)
memory usage: 58.7 KB


In [4]:
# 원본 데이터프레임 보존
data = df.copy()

# ip 컬럼 전처리 (str -> int)
data[['ip1', 'ip2', 'ip3', 'ip4']] = data['ip'].str.split('.', expand=True)
data[['ip1', 'ip2', 'ip3', 'ip4']] = data[['ip1', 'ip2', 'ip3', 'ip4']].astype(int)
data = data.drop('ip', axis=1)

# timestamp 에서 hour 추출 후 timestamp 제거
data['timestamp'] = pd.to_datetime(data['timestamp'])
data['hour'] = data['timestamp'].dt.hour
data = data.drop('timestamp', axis=1)

# status_code 를 통한 is_error 생성
data['is_error'] = data['status_code'].apply(
    lambda x: 1 if x >= 400 else 0
)

# label 생성 (에러인 코드이며, 크기가 2000바이트 이상인 로그)
data['label'] = np.where((data['is_error']==1) | (data['size'] > 2000), 1, 0)

# 범주형 데이터 원핫인코딩
data = pd.get_dummies(data,columns=['method']).astype(int)

# size 로그 변환
data['size'] = np.log1p(data['size'])

In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   status_code     1500 non-null   int64  
 1   size            1500 non-null   float64
 2   ip1             1500 non-null   int64  
 3   ip2             1500 non-null   int64  
 4   ip3             1500 non-null   int64  
 5   ip4             1500 non-null   int64  
 6   hour            1500 non-null   int64  
 7   is_error        1500 non-null   int64  
 8   label           1500 non-null   int64  
 9   method_DELETE   1500 non-null   int64  
 10  method_GET      1500 non-null   int64  
 11  method_OPTIONS  1500 non-null   int64  
 12  method_POST     1500 non-null   int64  
 13  method_PUT      1500 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 164.2 KB


In [6]:
data

,status_code,size,ip1,ip2,ip3,ip4,hour,is_error,label,method_DELETE,method_GET,method_OPTIONS,method_POST,method_PUT
0,301,6.295266,192,168,1,138,9,0,0,0,0,1,0,0
1,200,8.564077,192,168,1,130,12,0,1,0,0,1,0,0
2,200,8.661986,192,168,1,75,0,0,1,0,0,0,1,0
3,200,8.271804,192,168,1,176,19,0,1,1,0,0,0,0
4,200,8.957511,10,0,0,113,5,0,1,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,503,6.870053,192,168,1,65,5,1,1,0,1,0,0,0
1496,200,8.779711,192,168,1,51,17,0,1,0,0,1,0,0
1497,200,8.257126,10,0,0,116,21,0,1,0,0,1,0,0
1498,404,8.375399,192,168,1,2,8,1,1,0,1,0,0,0


In [ ]:
# label의 악성 요청 비율 확인
print(data['label'].value_counts())

label
1    1319
0     181
Name: count, dtype: int64


In [ ]:
# 피쳐, 타겟 설정 및 학습용 데이터 분리
X = data.drop(columns='label')
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42
)


      status_code      size  ip1  ip2  ip3  ip4  hour  is_error  \
0             301  6.295266  192  168    1  138     9         0   
1             200  8.564077  192  168    1  130    12         0   
2             200  8.661986  192  168    1   75     0         0   
3             200  8.271804  192  168    1  176    19         0   
4             200  8.957511   10    0    0  113     5         0   
...           ...       ...  ...  ...  ...  ...   ...       ...   
1495          503  6.870053  192  168    1   65     5         1   
1496          200  8.779711  192  168    1   51    17         0   
1497          200  8.257126   10    0    0  116    21         0   
1498          404  8.375399  192  168    1    2     8         1   
1499          503  9.109193   10    0    0    1    21         1   

      method_DELETE  method_GET  method_OPTIONS  method_POST  method_PUT  
0                 0           0               1            0           0  
1                 0           0              

In [9]:
# 모델 생성 및 학습
model = LogisticRegression(max_iter=10000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [11]:
# 평가 점수 계산
accuracy = accuracy_score(y_test, y_pred)   # 정확도 계산
precision = precision_score(y_test, y_pred) # 정밀도 계산
recall = recall_score(y_test, y_pred)       # 재현율 계산
f1 = f1_score(y_test, y_pred)               # F1-Score 계산

print('정확도(Accuracy):', accuracy)
print('정밀도(Precision):', precision)
print('재현율(Recall):', recall)
print('F1-Score:', f1)

print('========레포트========')
print(classification_report(y_test,y_pred))

정확도(Accuracy): 0.9766666666666667
정밀도(Precision): 0.9742647058823529
재현율(Recall): 1.0
F1-Score: 0.9869646182495344
========레포트========
              precision    recall  f1-score   support

           0       1.00      0.80      0.89        35
           1       0.97      1.00      0.99       265

    accuracy                           0.98       300
   macro avg       0.99      0.90      0.94       300
weighted avg       0.98      0.98      0.98       300



라벨을 오류 메시지 or 데이터 사이즈로 설정한 결과.
모델 성능이 전체적으로 좋은 편으로 나왔다. recall 이 1.0이 나와 악성 요청을 하나도 놓치지 않았다.
다만 이렇게 성능이 잘 나온 이유는 label을 만들 때 기준이 너무 명확했던 탓으로 보인다. 피쳐가 가진 힌트를 너무 가지고 있었던 탓인 것 같다.
실제 환경에서는 악성 코드 분류 기준이 더 복잡할 것이기 때문에 조금 더 정확한 학습이 가능할 것으로 보인다.
